In [1]:
import json
from pathlib import Path

# Path to MedQA
medqa_path = Path("../datasets/medqa/US/train.jsonl")

# Read first record
with open(medqa_path, "r", encoding="utf-8") as f:
    first_record = json.loads(f.readline())

first_record

{'question': 'A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physical exam is notable for an absence of costovertebral angle tenderness and a gravid uterus. Which of the following is the best treatment for this patient?',
 'answer': 'Nitrofurantoin',
 'options': {'A': 'Ampicillin',
  'B': 'Ceftriaxone',
  'C': 'Ciprofloxacin',
  'D': 'Doxycycline',
  'E': 'Nitrofurantoin'},
 'meta_info': 'step2&3',
 'answer_idx': 'E'}

In [2]:
# Count records
count = 0

with open(medqa_path, "r", encoding="utf-8") as f:
    for line in f:
        count += 1

print("Number of training records:", count)

Number of training records: 10178


In [3]:
from pathlib import Path

splits = ["train", "dev", "test"]

for split in splits:
    path = Path(f"../datasets/medqa/US/{split}.jsonl")

    count = 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            count += 1

    print(f"{split}: {count} records")

train: 10178 records
dev: 1272 records
test: 1273 records


In [4]:
import json

path = Path("../datasets/medqa/US/train.jsonl")

records = []

with open(path, "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

required_fields = [
    "question",
    "answer",
    "options",
    "meta_info",
    "answer_idx"
]

for field in required_fields:
    missing = sum(1 for r in records if field not in r or r[field] in [None, ""])
    print(f"{field}: {missing} missing")

question: 0 missing
answer: 0 missing
options: 0 missing
meta_info: 0 missing
answer_idx: 0 missing


In [5]:
from collections import Counter

answer_distribution = Counter(r["answer_idx"] for r in records)

print("Answer distribution:")
for option, count in sorted(answer_distribution.items()):
    print(f"{option}: {count}")

Answer distribution:
A: 2031
B: 2184
C: 2086
D: 2036
E: 1841


In [6]:
import json
from pathlib import Path

pubmedqa_path = Path("../datasets/pubmedqa/ori_pqal.json")

with open(pubmedqa_path, "r", encoding="utf-8") as f:
    pubmedqa_data = json.load(f)

print(type(pubmedqa_data))
print("Number of records:", len(pubmedqa_data))

<class 'dict'>
Number of records: 1000


In [7]:
# Display the first PubMedQA record

first_id = next(iter(pubmedqa_data))

print("First ID:", first_id)
print("\nFirst record:")
print(pubmedqa_data[first_id])

First ID: 21645374

First record:
{'QUESTION': 'Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?', 'CONTEXTS': ['Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.', 'The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cell

In [8]:
required_fields = [
    "QUESTION",
    "CONTEXTS",
    "LABELS",
    "MESHES",
    "YEAR",
    "reasoning_required_pred",
    "reasoning_free_pred",
    "final_decision",
    "LONG_ANSWER"
]

for field in required_fields:
    missing = sum(
        1 for record in pubmedqa_data.values()
        if field not in record or record[field] in [None, ""]
    )
    print(f"{field}: {missing} missing")

QUESTION: 0 missing
CONTEXTS: 0 missing
LABELS: 0 missing
MESHES: 0 missing
YEAR: 58 missing
reasoning_required_pred: 0 missing
reasoning_free_pred: 0 missing
final_decision: 0 missing
LONG_ANSWER: 0 missing


In [9]:
from collections import Counter

decision_distribution = Counter(
    record["final_decision"]
    for record in pubmedqa_data.values()
)

print("Final decision distribution:")

for decision, count in decision_distribution.items():
    print(f"{decision}: {count}")

Final decision distribution:
yes: 552
no: 338
maybe: 110


In [10]:
from pathlib import Path

medbullets_path = Path("../datasets/medbullets")

for file in medbullets_path.iterdir():
    print(file.name)

op4_eval-00000-of-00001.parquet
op4_test-00000-of-00001.parquet
op4_train-00000-of-00001.parquet
op5_eval-00000-of-00001.parquet
op5_test-00000-of-00001.parquet
op5_train-00000-of-00001.parquet


In [11]:
import pandas as pd
from pathlib import Path

medbullets_path = Path("../datasets/medbullets/op4_train-00000-of-00001.parquet")

medbullets_op4_train = pd.read_parquet(medbullets_path)

print("Shape:", medbullets_op4_train.shape)
print("\nColumns:")
print(medbullets_op4_train.columns.tolist())

Shape: (246, 6)

Columns:
['idx', 'question', 'options', 'answer', 'explanation', 'link']


In [12]:
medbullets_op4_train.head(1)

,idx,question,options,answer,explanation,link
0,001,A 17-year-old girl presents to the emergency d...,"{'A': 'Subarachnoid hemorrhage', 'B': 'Fat-sol...",B,"This patient is presenting with headache, bila...",https://bit.ly/3AgFkva


In [13]:
from pathlib import Path
import pandas as pd

medbullets_path = Path("../datasets/medbullets")

files = sorted(medbullets_path.glob("*.parquet"))

for file in files:
    df = pd.read_parquet(file)
    print(f"{file.name}: {len(df)} records")

op4_eval-00000-of-00001.parquet: 62 records
op4_test-00000-of-00001.parquet: 308 records
op4_train-00000-of-00001.parquet: 246 records
op5_eval-00000-of-00001.parquet: 62 records
op5_test-00000-of-00001.parquet: 308 records
op5_train-00000-of-00001.parquet: 246 records


In [14]:
op5_train_path = Path(
    "../datasets/medbullets/op5_train-00000-of-00001.parquet"
)

medbullets_op5_train = pd.read_parquet(op5_train_path)

print("Shape:", medbullets_op5_train.shape)
print("Columns:", medbullets_op5_train.columns.tolist())

medbullets_op5_train.head(1)

Shape: (246, 6)
Columns: ['idx', 'question', 'options', 'answer', 'explanation', 'link']


,idx,question,options,answer,explanation,link
0,001,A 17-year-old girl presents to the emergency d...,"{'A': 'Bacterial meningitis', 'B': 'Fat-solubl...",B,"This patient is presenting with headache, bila...",https://bit.ly/3AgFkva


In [15]:
# Check missing values in MedBullets OP4 and OP5

for version in ["op4", "op5"]:
    path = Path(
        f"../datasets/medbullets/{version}_train-00000-of-00001.parquet"
    )

    df = pd.read_parquet(path)

    print(f"\n{version.upper()} TRAIN")
    print("=" * 40)

    print(df.isnull().sum())


OP4 TRAIN
idx            0
question       0
options        0
answer         0
explanation    0
link           0
dtype: int64

OP5 TRAIN
idx            0
question       0
options        0
answer         0
explanation    0
link           0
dtype: int64


In [16]:
# Load OP4 and OP5 training sets

op4_path = Path(
    "../datasets/medbullets/op4_train-00000-of-00001.parquet"
)

op5_path = Path(
    "../datasets/medbullets/op5_train-00000-of-00001.parquet"
)

op4 = pd.read_parquet(op4_path)
op5 = pd.read_parquet(op5_path)

# Compare question IDs
same_ids = set(op4["idx"]) == set(op5["idx"])

# Compare question text
same_questions = set(op4["question"]) == set(op5["question"])

print("Same IDs:", same_ids)
print("Same questions:", same_questions)

Same IDs: True
Same questions: True


In [17]:
import json
import pandas as pd
from pathlib import Path

# Paths
raw_medqa = Path("../datasets/medqa/US")
processed_medqa = Path("../datasets/processed/medqa")

# Create output folder if it doesn't exist
processed_medqa.mkdir(parents=True, exist_ok=True)

# Read all MedQA splits
records = []

for split in ["train", "dev", "test"]:
    file_path = raw_medqa / f"{split}.jsonl"

    with open(file_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            record = json.loads(line)

            records.append({
                "sample_id": f"medqa_{split}_{i+1:05d}",
                "source_dataset": "MedQA",
                "split": split,
                "question": record["question"],
                "context": None,
                "options": record["options"],
                "ground_truth_answer": record["answer_idx"],
                "explanation": None,
                "source_reference": None,
                "metadata": record["meta_info"]
            })

# Convert to DataFrame
medqa_df = pd.DataFrame(records)

print("Total records:", len(medqa_df))
print("\nColumns:")
print(medqa_df.columns.tolist())

medqa_df.head()

Total records: 12723

Columns:
['sample_id', 'source_dataset', 'split', 'question', 'context', 'options', 'ground_truth_answer', 'explanation', 'source_reference', 'metadata']


,sample_id,source_dataset,split,question,context,options,ground_truth_answer,explanation,source_reference,metadata
0,medqa_train_00001,MedQA,train,A 23-year-old pregnant woman at 22 weeks gesta...,None,"{'A': 'Ampicillin', 'B': 'Ceftriaxone', 'C': '...",E,None,None,step2&3
1,medqa_train_00002,MedQA,train,A 3-month-old baby died suddenly at night whil...,None,{'A': 'Placing the infant in a supine position...,A,None,None,step2&3
2,medqa_train_00003,MedQA,train,A mother brings her 3-week-old infant to the p...,None,{'A': 'Abnormal migration of ventral pancreati...,A,None,None,step1
3,medqa_train_00004,MedQA,train,A pulmonary autopsy specimen from a 58-year-ol...,None,"{'A': 'Thromboembolism', 'B': 'Pulmonary ische...",A,None,None,step1
4,medqa_train_00005,MedQA,train,A 20-year-old woman presents with menorrhagia ...,None,"{'A': 'Factor V Leiden', 'B': 'Hemophilia A', ...",E,None,None,step1


In [18]:
output_file = processed_medqa / "medqa_normalized.csv"

medqa_df.to_csv(output_file, index=False)

print(f"Saved to: {output_file}")

Saved to: ..\datasets\processed\medqa\medqa_normalized.csv


In [19]:
# Validate the normalized MedQA file

normalized_medqa = pd.read_csv(
    "../datasets/processed/medqa/medqa_normalized.csv"
)

print("Shape:", normalized_medqa.shape)

print("\nColumns:")
print(normalized_medqa.columns.tolist())

print("\nRecords by split:")
print(normalized_medqa["split"].value_counts())

print("\nMissing values:")
print(normalized_medqa.isnull().sum())

Shape: (12723, 10)

Columns:
['sample_id', 'source_dataset', 'split', 'question', 'context', 'options', 'ground_truth_answer', 'explanation', 'source_reference', 'metadata']

Records by split:
split
train    10178
test      1273
dev       1272
Name: count, dtype: int64

Missing values:
sample_id                  0
source_dataset             0
split                      0
question                   0
context                12723
options                    0
ground_truth_answer        0
explanation            12723
source_reference       12723
metadata                   0
dtype: int64


In [20]:
# PubMedQA normalization

import json
import pandas as pd
from pathlib import Path

# Paths
raw_pubmedqa = Path("../datasets/pubmedqa/ori_pqal.json")
processed_pubmedqa = Path("../datasets/processed/pubmedqa")

# Create output folder
processed_pubmedqa.mkdir(parents=True, exist_ok=True)

# Load PubMedQA
with open(raw_pubmedqa, "r", encoding="utf-8") as f:
    pubmedqa_data = json.load(f)

records = []

for pmid, record in pubmedqa_data.items():

    records.append({
        "sample_id": f"pubmedqa_{pmid}",
        "source_dataset": "PubMedQA",
        "split": "labeled",
        "question": record["QUESTION"],
        "context": record["CONTEXTS"],
        "options": {
            "yes": "yes",
            "no": "no",
            "maybe": "maybe"
        },
        "ground_truth_answer": record["final_decision"],
        "explanation": record["LONG_ANSWER"],
        "source_reference": pmid,
        "metadata": {
            "labels": record["LABELS"],
            "meshes": record["MESHES"],
            "year": record["YEAR"],
            "reasoning_required_pred": record["reasoning_required_pred"],
            "reasoning_free_pred": record["reasoning_free_pred"]
        }
    })

pubmedqa_df = pd.DataFrame(records)

print("Shape:", pubmedqa_df.shape)
print("\nColumns:")
print(pubmedqa_df.columns.tolist())

pubmedqa_df.head(1)

Shape: (1000, 10)

Columns:
['sample_id', 'source_dataset', 'split', 'question', 'context', 'options', 'ground_truth_answer', 'explanation', 'source_reference', 'metadata']


,sample_id,source_dataset,split,question,context,options,ground_truth_answer,explanation,source_reference,metadata
0,pubmedqa_21645374,PubMedQA,labeled,Do mitochondria play a role in remodelling lac...,[Programmed cell death (PCD) is the regulated ...,"{'yes': 'yes', 'no': 'no', 'maybe': 'maybe'}",yes,Results depicted mitochondrial dynamics in viv...,21645374,"{'labels': ['BACKGROUND', 'RESULTS'], 'meshes'..."


In [21]:
output_file = processed_pubmedqa / "pubmedqa_normalized.csv"

pubmedqa_df.to_csv(output_file, index=False)

print(f"Saved to: {output_file}")

Saved to: ..\datasets\processed\pubmedqa\pubmedqa_normalized.csv


In [22]:
# Validate normalized PubMedQA

normalized_pubmedqa = pd.read_csv(
    "../datasets/processed/pubmedqa/pubmedqa_normalized.csv"
)

print("Shape:", normalized_pubmedqa.shape)

print("\nColumns:")
print(normalized_pubmedqa.columns.tolist())

print("\nMissing values:")
print(normalized_pubmedqa.isnull().sum())

print("\nDecision distribution:")
print(normalized_pubmedqa["ground_truth_answer"].value_counts())

Shape: (1000, 10)

Columns:
['sample_id', 'source_dataset', 'split', 'question', 'context', 'options', 'ground_truth_answer', 'explanation', 'source_reference', 'metadata']

Missing values:
sample_id              0
source_dataset         0
split                  0
question               0
context                0
options                0
ground_truth_answer    0
explanation            0
source_reference       0
metadata               0
dtype: int64

Decision distribution:
ground_truth_answer
yes      552
no       338
maybe    110
Name: count, dtype: int64


In [23]:
# MedBullets normalization

import pandas as pd
from pathlib import Path

# Paths
raw_medbullets = Path("../datasets/medbullets")
processed_medbullets = Path("../datasets/processed/medbullets")

# Create output folder
processed_medbullets.mkdir(parents=True, exist_ok=True)

records = []

# Process OP4 splits
for split in ["train", "eval", "test"]:

    file_path = raw_medbullets / f"op4_{split}-00000-of-00001.parquet"

    df = pd.read_parquet(file_path)

    for _, row in df.iterrows():

        records.append({
            "sample_id": f"medbullets_op4_{split}_{row['idx']}",
            "source_dataset": "MedBullets",
            "split": split,
            "question": row["question"],
            "context": None,
            "options": row["options"],
            "ground_truth_answer": row["answer"],
            "explanation": row["explanation"],
            "source_reference": row["link"],
            "metadata": {
                "version": "op4",
                "original_id": row["idx"]
            }
        })

# Create DataFrame
medbullets_df = pd.DataFrame(records)

print("Shape:", medbullets_df.shape)
print("\nColumns:")
print(medbullets_df.columns.tolist())

medbullets_df.head(1)

Shape: (616, 10)

Columns:
['sample_id', 'source_dataset', 'split', 'question', 'context', 'options', 'ground_truth_answer', 'explanation', 'source_reference', 'metadata']


,sample_id,source_dataset,split,question,context,options,ground_truth_answer,explanation,source_reference,metadata
0,medbullets_op4_train_001,MedBullets,train,A 17-year-old girl presents to the emergency d...,None,"{'A': 'Subarachnoid hemorrhage', 'B': 'Fat-sol...",B,"This patient is presenting with headache, bila...",https://bit.ly/3AgFkva,"{'version': 'op4', 'original_id': '001'}"


In [24]:
output_file = processed_medbullets / "medbullets_op4_normalized.csv"

medbullets_df.to_csv(output_file, index=False)

print(f"Saved to: {output_file}")

Saved to: ..\datasets\processed\medbullets\medbullets_op4_normalized.csv


In [25]:
# Validate normalized MedBullets

normalized_medbullets = pd.read_csv(
    "../datasets/processed/medbullets/medbullets_op4_normalized.csv"
)

print("Shape:", normalized_medbullets.shape)

print("\nColumns:")
print(normalized_medbullets.columns.tolist())

print("\nRecords by split:")
print(normalized_medbullets["split"].value_counts())

print("\nMissing values:")
print(normalized_medbullets.isnull().sum())

print("\nAnswer distribution:")
print(normalized_medbullets["ground_truth_answer"].value_counts())

Shape: (616, 10)

Columns:
['sample_id', 'source_dataset', 'split', 'question', 'context', 'options', 'ground_truth_answer', 'explanation', 'source_reference', 'metadata']

Records by split:
split
test     308
train    246
eval      62
Name: count, dtype: int64

Missing values:
sample_id                0
source_dataset           0
split                    0
question                 0
context                616
options                  0
ground_truth_answer      0
explanation              0
source_reference         0
metadata                 0
dtype: int64

Answer distribution:
ground_truth_answer
A    174
C    154
B    152
D    136
Name: count, dtype: int64


In [26]:
print(
    normalized_medbullets["ground_truth_answer"]
    .value_counts()
    .sort_index()
)

ground_truth_answer
A    174
B    152
C    154
D    136
Name: count, dtype: int64
